In [1]:
LEAGUE = "brasileirao-serie-a"
SEASON = "2025"
ENVIRONMENT = "prd"
REPROC_MODE = True

In [2]:
import boto3
import json
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("silver") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://cvmc-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

s3 = boto3.client(
    's3',
    endpoint_url='http://cvmc-minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin'
)

print(f"Spark version: {spark.version}")
print("Delta + MinIO configurados!")

Spark version: 3.5.0
Delta + MinIO configurados!


In [3]:
# =====================
# PARÂMETROS DE EXECUÇÃO
# =====================
# LEAGUE = "brasileirao-serie-a"
# SEASON = "2026"
# ENVIRONMENT = "dev"
# REPROC_MODE = False

# =====================
# CONFIGURAÇÕES FIXAS
# =====================
BUCKET = "eng-prd-data-lake"
RAW_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/raw/{LEAGUE}"
SILVER_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/silver/{LEAGUE}"

DEDUP_KEYS = {
    "standing": {
        "pk": ["season", "record.teamId"],
        "order_col": "ingested_at"
    },
    "rounds": {
        "pk": ["record.id"],
        "order_col": "ingested_at"
    },
    "events": {
        "pk": ["record.events.id"],
        "order_col": "ingested_at"
    },
    "players": {
        "pk": ["record.players.id", "record.teams.teamId", "season"],
        "order_col": "ingested_at"
    },
    "team_stats": {
        "pk": ["record.team.teamId", "season"],
        "order_col": "ingested_at"
    },
    "player_stats": {
    "pk": ["record.team.teamId", "season"],
    "order_col": "ingested_at"
    }
}

print(f"League: {LEAGUE} | Season: {SEASON} | Env: {ENVIRONMENT} | Reproc: {REPROC_MODE}")
print(f"Raw:    {RAW_PATH}")
print(f"Silver: {SILVER_PATH}")

League: brasileirao-serie-a | Season: 2025 | Env: prd | Reproc: True
Raw:    s3a://eng-prd-data-lake/prd/raw/brasileirao-serie-a
Silver: s3a://eng-prd-data-lake/prd/silver/brasileirao-serie-a


In [4]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

def raw_to_silver(data_type):
    raw = f"{RAW_PATH}/{data_type}"
    silver = f"{SILVER_PATH}/{data_type}"
    config = DEDUP_KEYS[data_type]
    pks = config["pk"]
    order_col = config["order_col"]

    print(f"Lendo {data_type} da Raw...")
    
    df = spark.read.format("delta").load(raw)
    print(raw)
    # Deduplicação — mantém o registro mais recente por chave
    window = Window.partitionBy([col(pk) for pk in pks]) \
                   .orderBy(col(order_col).desc())

    df_dedup = df.withColumn("_rank", row_number().over(window)) \
                 .filter(col("_rank") == 1) \
                 .drop("_rank")

    print(f"Registros após dedup: {df_dedup.count()}")

    df_dedup.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("season") \
        .save(silver)

    print(f"[OK] {data_type} salvo na Silver!\n")

for data_type in ["standing", "rounds", "events", "players", "team_stats", "player_stats"]:
    raw_to_silver(data_type)

print("Camada Silver finalizada!")

Lendo standing da Raw...
s3a://eng-prd-data-lake/prd/raw/brasileirao-serie-a/standing
Registros após dedup: 60
[OK] standing salvo na Silver!

Lendo rounds da Raw...
s3a://eng-prd-data-lake/prd/raw/brasileirao-serie-a/rounds
Registros após dedup: 38
[OK] rounds salvo na Silver!

Lendo events da Raw...
s3a://eng-prd-data-lake/prd/raw/brasileirao-serie-a/events
Registros após dedup: 20
[OK] events salvo na Silver!

Lendo players da Raw...
s3a://eng-prd-data-lake/prd/raw/brasileirao-serie-a/players
Registros após dedup: 1098
[OK] players salvo na Silver!

Lendo team_stats da Raw...
s3a://eng-prd-data-lake/prd/raw/brasileirao-serie-a/team_stats
Registros após dedup: 60
[OK] team_stats salvo na Silver!

Lendo player_stats da Raw...
s3a://eng-prd-data-lake/prd/raw/brasileirao-serie-a/player_stats
Registros após dedup: 60
[OK] player_stats salvo na Silver!

Camada Silver finalizada!


In [5]:
import psycopg2

PG_URL = "jdbc:postgresql://cvmc-postgres:5432/cvmc_data?currentSchema=silver"
PG_PROPERTIES = {
    "user": "airflow",
    "password": "airflow",
    "driver": "org.postgresql.Driver"
}
PG_CONN = {
    "host": "cvmc-postgres",
    "port": 5432,
    "dbname": "cvmc_data",
    "user": "airflow",
    "password": "airflow"
}

# Garante que o schema silver existe
conn = psycopg2.connect(**PG_CONN)
conn.autocommit = True
cur = conn.cursor()
cur.execute("CREATE SCHEMA IF NOT EXISTS silver;")
cur.close()
conn.close()
print("Schema silver garantido!")

Schema silver garantido!


In [6]:
import psycopg2
from pyspark.sql.functions import to_json, col, current_timestamp, from_utc_timestamp

def save_silver_to_postgres(data_type):
    print(f"Processando {data_type}...")
    table_name = f"{data_type}"

    # Lê a Silver (Delta)
    df = spark.read.format("delta").load(f"{SILVER_PATH}/{data_type}")

    # Converte record (struct) para JSONB, remove ingested_at, adiciona dt_updated_at
    df_pg = df.withColumn("record", to_json(col("record"))) \
              .drop("ingested_at") \
              .withColumn(
                  "dt_updated_at",
                  from_utc_timestamp(current_timestamp(), "America/Sao_Paulo")
              )

    # Escreve no Postgres (record entra como TEXT)
    df_pg.write.jdbc(
        url=PG_URL,
        table=table_name,
        mode="overwrite",
        properties=PG_PROPERTIES
    )

    # Converte a coluna record de TEXT para JSONB
    conn = psycopg2.connect(**PG_CONN)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(f"""
        ALTER TABLE silver.{table_name}
        ALTER COLUMN record TYPE JSONB
        USING record::jsonb;
    """)
    cur.close()
    conn.close()

    print(f"[OK] {table_name} salvo no schema silver como JSONB! Registros: {df_pg.count()}")


for data_type in ["standing", "rounds", "events", "players", "team_stats", "player_stats"]:
    save_silver_to_postgres(data_type)

print("\nSilver no Postgres finalizada!")

Processando standing...
[OK] standing salvo no schema silver como JSONB! Registros: 60
Processando rounds...
[OK] rounds salvo no schema silver como JSONB! Registros: 38
Processando events...
[OK] events salvo no schema silver como JSONB! Registros: 20
Processando players...
[OK] players salvo no schema silver como JSONB! Registros: 1098
Processando team_stats...
[OK] team_stats salvo no schema silver como JSONB! Registros: 60
Processando player_stats...
[OK] player_stats salvo no schema silver como JSONB! Registros: 60

Silver no Postgres finalizada!
